## Load datasets

In [7]:
from datasets import load_dataset
from tqdm import tqdm
import random 

# Download 10,000 paragraphs (passages) to use for knowledge base
# Use 'mteb/msmarco' as it has been handled for benchmark 
dataset = load_dataset("mteb/msmarco", "corpus", split="corpus", streaming=True)
categories = ["finance", "legal", "tech", "medical"]
total_corpus = []
for i, item in tqdm(enumerate(dataset)):
    if i >= 2000000:
        break
    total_corpus.append({"id": i, 
                    "text": item["text"],
                    "category": random.choice(categories),
                    "is_public": i%2==0})

# Download question set (queries) correspond to datasets
queries_ds = load_dataset("mteb/msmarco", "queries", split="queries", streaming=True)
total_test_queries = []
for i, item in tqdm(enumerate(queries_ds)):
    if i >= 10000:
        break
    total_test_queries.append(item["text"])

d:\Vector DB Comparison\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2000000it [06:57, 4794.21it/s] 
10000it [00:05, 1980.16it/s]


In [8]:
corpus, update_corpus = total_corpus[:10000], total_corpus[10000:20000]
test_queries, test_query = total_test_queries[:1000], total_corpus[10001]

print(corpus[0]) #10000
print(f"{test_queries[0]} \n") #1000
print(test_query)

{'id': 0, 'text': 'The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.', 'category': 'finance', 'is_public': True}
)what was the immediate impact of the success of the manhattan project? 

{'id': 10001, 'text': 'Hubs and switches serve as a central connection for all of your network equipment and handles a data type known as frames. Frames carry your data. When a frame is received, it is amplified and then transmitted on to the port of the destination PC.', 'category': 'finance', 'is_public': False}


## Connect to database

In [2]:
import os 
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
#Initialize Qdrant vector database 
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams


qdrant = QdrantClient(url=os.getenv("qdrant_endpoint"),
                    api_key=os.getenv("qdrant_api_key"),
                    timeout=60) 

In [ ]:
qdrant.recreate_collection(
    collection_name="rag_test",
    vectors_config=VectorParams(size=768, distance=Distance.COSINE) # Assume model 768D
)

In [7]:
#Initialize Milvus
from pymilvus import MilvusClient


milvus = MilvusClient(uri=os.getenv("milvus_endpoint"),
                        token=os.getenv("milvus_token"))

In [ ]:
milvus.create_collection(
    collection_name="rag_test",
    dimension=768
)

In [3]:
import weaviate 

auth_config = weaviate.auth.AuthApiKey(api_key=os.getenv("weaviate_api_key"))

weaviate_client = weaviate.connect_to_weaviate_cloud(cluster_url=os.getenv("weaviate_endpoint"),
                                                    auth_credentials=auth_config)

In [9]:
from pinecone import Pinecone

pinecone= Pinecone(api_key=os.getenv("pinecone_api_key"))
pinecone_index = pinecone.Index('test')

In [10]:
import chromadb

chroma = chromadb.CloudClient(
  api_key=os.getenv("chroma_api_key"),
  tenant=os.getenv("chroma_tenant"),
  database='test'
)

In [11]:
from fastembed import TextEmbedding
import numpy as np

model = TextEmbedding(model_name="BAAI/bge-base-en-v1.5")

## Embedding process 

In [ ]:
#run this for the first time
documents = [item["text"] for item in corpus]

embeddings_generator = model.embed(documents,batch_size=128)
all_embeddings = np.array(list(embeddings_generator))

np.save("msmarco_vectors_768.npy", all_embeddings)

In [4]:
#Run this cell for next times
import numpy as np 
all_embeddings = np.load("msmarco_vectors_768.npy")

In [11]:
print(all_embeddings[0])

[-1.96880568e-02  1.10764224e-02 -4.43056896e-02  6.17255382e-02
  2.55551599e-02 -6.12889067e-04  2.29089148e-02  2.37557124e-02
 -4.81616473e-03 -4.95679341e-02 -2.98798773e-02  2.71429052e-03
 -2.59785578e-02  7.65747502e-02 -3.29041556e-02  7.03447312e-02
  3.65332924e-02 -4.09033755e-03 -1.22332089e-02  7.53650367e-02
 -2.94413585e-02 -1.57262515e-02  3.90434451e-02  3.25714871e-02
  3.20271179e-02 -1.51213957e-03 -4.97871963e-03 -6.41752034e-02
 -2.63565928e-02  2.21226029e-02  3.73498462e-02 -2.15782318e-02
 -4.33379188e-02 -3.72288749e-02 -1.73593629e-02 -6.92559918e-03
  3.27567244e-03  3.22690606e-02 -1.54805288e-03 -2.55740597e-03
 -1.43426443e-02 -1.46601936e-02 -2.30147652e-02 -4.81994497e-03
 -1.07240938e-01 -8.31676740e-03 -5.52233383e-02  5.84895574e-02
  3.34324606e-04  2.36838870e-03 -3.41743529e-02  3.79736046e-03
  3.96482982e-02  2.09733769e-02  4.82750544e-03  4.01624292e-02
 -2.99706068e-02 -6.92181895e-03 -4.23096642e-02 -5.02937622e-02
  1.47509221e-02 -1.76731

## Ingesting process 

In [5]:
# --- 1. Milvus Bulk Insert ---
def bulk_insert_milvus(corpus, embeddings, collection_name="rag_test"):
    milvus_data = [
        {
            "id": item["id"], 
            "vector": emb.tolist() if hasattr(emb, "tolist") else emb, 
            "text": item["text"], 
            "category": item["category"], 
            "is_public": item["is_public"]
        }
        for item, emb in zip(corpus, embeddings)
    ]
    for i in range(0, len(milvus_data), 500):
        milvus.insert(collection_name=collection_name, data=milvus_data[i : i+500])
    print("Milvus done")

# --- 2. Qdrant Bulk Insert ---
def bulk_insert_qdrant(corpus, embeddings, collection_name="rag_test"):
    from qdrant_client.models import PointStruct
    qdrant_points = [
        PointStruct(
            id=item["id"], 
            vector=emb.tolist() if hasattr(emb, "tolist") else emb, 
            payload={"text": item["text"], "category": item["category"], "is_public": item["is_public"]}
        )
        for item, emb in zip(corpus, embeddings)
    ]
    qdrant.upload_points(collection_name=collection_name, points=qdrant_points, batch_size=100)
    print("Qdrant done")

# --- 3. Weaviate Bulk Insert ---
def bulk_insert_weaviate(corpus, embeddings, collection_name="RagTest"):
    weaviate_coll = weaviate_client.collections.get(collection_name)
    with weaviate_coll.batch.dynamic() as batch:
        for item, emb in zip(corpus, embeddings):
            batch.add_object(
                properties={
                    "doc_id": item["id"], 
                    "text": item["text"], 
                    "category": item["category"], 
                    "is_public": item["is_public"]
                },
                vector=emb.tolist() if hasattr(emb, "tolist") else emb
            )
    print("Weaviate done")

# --- 4. Pinecone Bulk Insert ---
def bulk_insert_pinecone(corpus, embeddings, index_name="test"):
    pinecone_data = [
        (
            str(item["id"]), 
            emb.tolist() if hasattr(emb, "tolist") else emb, 
            {"text": item["text"], "category": item["category"], "is_public": item["is_public"], "doc_id": item["id"]}
        )
        for item, emb in zip(corpus, embeddings)
    ]

    for i in range(0, len(pinecone_data), 100):
        pinecone_index.upsert(vectors=pinecone_data[i : i+100])
    print("Pinecone done")

# --- 5. Chroma Bulk Insert ---
def bulk_insert_chroma(corpus, embeddings, collection_name="RagTest"):
    chroma_coll = chroma.get_or_create_collection(collection_name)
    
    ids_str = [str(item["id"]) for item in corpus]
    docs_str = [item["text"] for item in corpus]
    meta_list = [{"category": item["category"], "is_public": item["is_public"], "doc_id": item["id"]} for item in corpus]
    emb_list = [emb.tolist() if hasattr(emb, "tolist") else emb for emb in embeddings]

    for i in range(0, len(ids_str), 100): 
        chroma_coll.add(
            ids=ids_str[i:i+100], 
            embeddings=emb_list[i:i+100], 
            metadatas=meta_list[i:i+100], 
            documents=docs_str[i:i+100]
        )
    print("Chroma done")


In [6]:
weaviate_collection = weaviate_client.collections.get("RagTest")

In [13]:
chroma_collection = chroma.get_or_create_collection("RagTest")

In [ ]:
bulk_insert_chroma(corpus,all_embeddings)
bulk_insert_milvus(corpus,all_embeddings)
bulk_insert_weaviate(corpus,all_embeddings)
bulk_insert_qdrant(corpus,all_embeddings)
bulk_insert_pinecone(corpus,all_embeddings)

In [16]:
import faiss

#Use FAISS as a baseline to evaluate other vector database 
d = 768
index_flat = faiss.IndexFlatL2(d)
index_flat.add(all_embeddings.astype('float32'))

metadata_store = {item["id"]: {"category": item["category"], "is_public": item["is_public"]} for item in corpus}

def get_ground_truth(query_vector, k=10, level=0, **kwargs):
    valid_ids = []
    
    for db_id, meta in metadata_store.items():
        if level == 0:
            valid_ids.append(db_id)
            
        elif level == 1:
            if meta['is_public'] == True:
                valid_ids.append(db_id)
                
        elif level == 2:
            if db_id < 1000:
                valid_ids.append(db_id)
                
        elif level == 3:
            cat_target = kwargs.get('target_category', 'finance')
            if meta['category'] == cat_target and meta['is_public'] == True:
                valid_ids.append(db_id)

    if len(valid_ids) == 0:
        return []
        
    valid_ids_np = np.array(valid_ids, dtype=np.int64)
    id_selector = faiss.IDSelectorBatch(valid_ids_np)
    search_params = faiss.SearchParameters(sel=id_selector)

    distances, indices = index_flat.search(
        query_vector.reshape(1, -1).astype('float32'), 
        k,
        params=search_params
    )
    
    return [idx for idx in indices[0].tolist() if idx != -1]

<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


## Measure Recall@10 and latency with no filter (level 0)

In [24]:
#Send queries to each database and get top-10 results (ids) and compare to FAISS results 
import numpy as np
import time 

def calculate_recall(database_results,ground_truth_ids):
    if len(ground_truth_ids) == 0:
        return None
    intersection = set(database_results).intersection(set(ground_truth_ids))
    return len(intersection)/len(set(ground_truth_ids))

# Warm-up to cache/reconnect as most vector DB in Cloud take some time to handshake/TCP connection and cache into RAM 
for i in range(5):
    query_vec = list(model.embed([test_queries[i]]))[0].tolist()

    milvus_results = milvus.search(collection_name="rag_test", data = [query_vec], limit=10,output_fields=['id']) #3-D
    qdrant_hits = qdrant.query_points(collection_name="rag_test",query=query_vec,limit=10)
    weaviate_res = weaviate_collection.query.near_vector(near_vector=query_vec, limit=10, return_properties=["doc_id"])
    pinecone_res = pinecone_index.query(vector=query_vec, top_k=10, include_metadata=False, include_values=False)
    chroma_res = chroma_collection.query(query_embeddings=[query_vec], n_results=10, include=[]) # Only get ID

In [48]:
#Measure recall@10 and latency
import time 

all_recalls = {"qdrant":[], "pinecone":[], "milvus":[], "weaviate":[], "chroma":[]}
latencies = {"qdrant":[], "pinecone":[], "milvus":[], "weaviate":[], "chroma":[]}

for i in range(len(test_queries)):
    query_vec = list(model.embed([test_queries[i]]))[0].tolist()

    gt_ids = get_ground_truth(np.array(query_vec))
    
    start = time.perf_counter()
    milvus_results = milvus.search(collection_name="rag_test", data = [query_vec], limit=10,output_fields=['id']) #3-D
    end = time.perf_counter()
    milvus_ids = [hit["id"] for hit in milvus_results[0]]
    all_recalls["milvus"].append(calculate_recall(milvus_ids,gt_ids))
    latencies["milvus"].append(end-start)
    
    start = time.perf_counter()
    qdrant_hits = qdrant.query_points(collection_name="rag_test",query=query_vec,limit=10)
    end = time.perf_counter()
    qdrant_ids = [hit.id for hit in qdrant_hits.points]
    all_recalls["qdrant"].append(calculate_recall(qdrant_ids,gt_ids))
    latencies["qdrant"].append(end-start)

    start = time.perf_counter()
    weaviate_res = weaviate_collection.query.near_vector(near_vector=query_vec, limit=10, return_properties=["doc_id"])
    end = time.perf_counter()
    weaviate_ids = [obj.properties["doc_id"] for obj in weaviate_res.objects]
    all_recalls["weaviate"].append(calculate_recall(weaviate_ids,gt_ids))
    latencies["weaviate"].append(end-start)

    start =time.perf_counter()
    pinecone_res = pinecone_index.query(vector=query_vec, top_k=10, include_metadata=False, include_values=False)
    end = time.perf_counter()
    pinecone_ids = [int(match.id) for match in pinecone_res["matches"]]
    all_recalls["pinecone"].append(calculate_recall(pinecone_ids,gt_ids))
    latencies["pinecone"].append(end-start)

    start = time.perf_counter()
    chroma_res = chroma_collection.query(query_embeddings=[query_vec], n_results=10, include=[]) # Only get ID
    end = time.perf_counter()
    chroma_ids = [int(i) for i in chroma_res["ids"][0]]
    all_recalls["chroma"].append(calculate_recall(chroma_ids,gt_ids))
    latencies["chroma"].append(end-start)

In [49]:
for db,recall in all_recalls.items():
    filter_recall = [v for v in recall if v is not None]
    print(f"Average recall of {db}: {100*np.mean(filter_recall):.2f}%")

Average recall of qdrant: 99.78%
Average recall of pinecone: 99.04%
Average recall of milvus: 95.49%
Average recall of weaviate: 99.73%
Average recall of chroma: 98.67%


In [50]:
for db, latency in latencies.items():
    p50 = np.percentile(latency, 50)
    p95 = np.percentile(latency, 95)
    p99 = np.percentile(latency, 99)
    mean = np.mean(latency)

    print(f"{db} DB:")
    print(f"Baseline Latency (P99): {p99:.2f} ms")
    print(f"Median Latency (P50): {p50:.2f} ms")
    print(f"Mean Latency (Mean): {mean:.2f} ms")
    print(f"\n")

qdrant DB:
Baseline Latency (P99): 0.28 ms
Median Latency (P50): 0.27 ms
Mean Latency (Mean): 0.27 ms


pinecone DB:
Baseline Latency (P99): 0.27 ms
Median Latency (P50): 0.24 ms
Mean Latency (Mean): 0.24 ms


milvus DB:
Baseline Latency (P99): 0.25 ms
Median Latency (P50): 0.23 ms
Mean Latency (Mean): 0.23 ms


weaviate DB:
Baseline Latency (P99): 0.10 ms
Median Latency (P50): 0.08 ms
Mean Latency (Mean): 0.08 ms


chroma DB:
Baseline Latency (P99): 0.30 ms
Median Latency (P50): 0.26 ms
Mean Latency (Mean): 0.26 ms




## Recall@10 and Latency with filter (level 1)

In [29]:
from qdrant_client.models import PayloadSchemaType
# Create index for is_public as Qdrant can not create by itself
qdrant.create_payload_index(
    collection_name="rag_test",
    field_name="is_public",
    field_schema=PayloadSchemaType.BOOL  
)
# Create index for category
qdrant.create_payload_index(
    collection_name="rag_test",
    field_name="category",
    field_schema=PayloadSchemaType.KEYWORD  
)

from qdrant_client.models import Filter as QdrantFilter, FieldCondition, MatchValue
qdrant_filter = QdrantFilter(must=[FieldCondition(key="is_public", match=MatchValue(value=True))])

In [30]:
from weaviate.classes.query import Filter as WeaviateFilter

all_recalls_2 = {"qdrant":[], "pinecone":[], "milvus":[], "weaviate":[], "chroma":[]}
latencies_2 = {"qdrant":[], "pinecone":[], "milvus":[], "weaviate":[], "chroma":[]}


#Query with filter is_public == True
for i in range(len(test_queries)):
    query_vec = list(model.embed([test_queries[i]]))[0].tolist()

    gt_ids = get_ground_truth(np.array(query_vec),10,1)
    
    start = time.perf_counter()
    milvus_results = milvus.search(collection_name="rag_test", data = [query_vec], limit=10,output_fields=['id'],filter="is_public == true") #3-D
    end = time.perf_counter()
    milvus_ids = [hit["id"] for hit in milvus_results[0]]
    all_recalls_2["milvus"].append(calculate_recall(milvus_ids,gt_ids))
    latencies_2["milvus"].append(end-start)
    
    start = time.perf_counter()
    qdrant_hits = qdrant.query_points(collection_name="rag_test",query=query_vec,limit=10,query_filter=qdrant_filter)
    end = time.perf_counter()
    qdrant_ids = [hit.id for hit in qdrant_hits.points]
    all_recalls_2["qdrant"].append(calculate_recall(qdrant_ids,gt_ids))
    latencies_2["qdrant"].append(end-start)

    start = time.perf_counter()
    weaviate_res = weaviate_collection.query.near_vector(near_vector=query_vec, limit=10, return_properties=["doc_id"],filters=WeaviateFilter.by_property("is_public").equal(True))
    end = time.perf_counter()
    weaviate_ids = [obj.properties["doc_id"] for obj in weaviate_res.objects]
    all_recalls_2["weaviate"].append(calculate_recall(weaviate_ids,gt_ids))
    latencies_2["weaviate"].append(end-start)

    start =time.perf_counter()
    pinecone_res = pinecone_index.query(vector=query_vec, top_k=10, include_metadata=False, include_values=False, filter={"is_public": {"$eq": True}})
    end = time.perf_counter()
    pinecone_ids = [int(match.id) for match in pinecone_res["matches"]]
    all_recalls_2["pinecone"].append(calculate_recall(pinecone_ids,gt_ids))
    latencies_2["pinecone"].append(end-start)

    start = time.perf_counter()
    chroma_res = chroma_collection.query(query_embeddings=[query_vec], n_results=10, include=[], where={"is_public": {"$eq": True}} ) # Only get ID
    end = time.perf_counter()
    chroma_ids = [int(i) for i in chroma_res["ids"][0]]
    all_recalls_2["chroma"].append(calculate_recall(chroma_ids,gt_ids))
    latencies_2["chroma"].append(end-start)

In [31]:
for db, recall in all_recalls_2.items():
    filter_recall = [v for v in recall if v is not None]
    print(f"Average recall of {db}: {100*np.mean(filter_recall):.2f}%")

Average recall of qdrant: 98.14%
Average recall of pinecone: 98.87%
Average recall of milvus: 95.60%
Average recall of weaviate: 100.00%
Average recall of chroma: 98.24%


In [32]:
for db, latency in latencies_2.items():
    p50 = np.percentile(latency, 50)
    p95 = np.percentile(latency, 95)
    p99 = np.percentile(latency, 99)
    mean = np.mean(latency)

    print(f"{db} DB:")
    print(f"Baseline Latency (P99): {p99:.2f} ms")
    print(f"Median Latency (P50): {p50:.2f} ms")
    print(f"Mean Latency (Mean): {mean:.2f} ms")
    print(f"\n")

qdrant DB:
Baseline Latency (P99): 0.28 ms
Median Latency (P50): 0.27 ms
Mean Latency (Mean): 0.27 ms


pinecone DB:
Baseline Latency (P99): 0.27 ms
Median Latency (P50): 0.25 ms
Mean Latency (Mean): 0.25 ms


milvus DB:
Baseline Latency (P99): 0.26 ms
Median Latency (P50): 0.24 ms
Mean Latency (Mean): 0.24 ms


weaviate DB:
Baseline Latency (P99): 0.10 ms
Median Latency (P50): 0.08 ms
Mean Latency (Mean): 0.08 ms


chroma DB:
Baseline Latency (P99): 0.30 ms
Median Latency (P50): 0.25 ms
Mean Latency (Mean): 0.26 ms




## Recall@10 and Latency with filter (level 2)

In [39]:
from qdrant_client.models import HasIdCondition

valid_ids = list(range(1000))
all_recalls_3 = {"qdrant":[], "pinecone":[], "milvus":[], "weaviate":[], "chroma":[]}
latencies_3 = {"qdrant":[], "pinecone":[], "milvus":[], "weaviate":[], "chroma":[]}


#Query with filter id < 1000
for i in range(len(test_queries)):
    query_vec = list(model.embed([test_queries[i]]))[0].tolist()

    gt_ids = get_ground_truth(np.array(query_vec),10,2)
    
    start = time.perf_counter()
    milvus_results = milvus.search(collection_name="rag_test", data = [query_vec], limit=10,output_fields=['id'],filter="id < 1000") #3-D
    end = time.perf_counter()
    milvus_ids = [hit["id"] for hit in milvus_results[0]]
    all_recalls_3["milvus"].append(calculate_recall(milvus_ids,gt_ids))
    latencies_3["milvus"].append(end-start)
    
    start = time.perf_counter()
    qdrant_hits = qdrant.query_points(collection_name="rag_test",query=query_vec,limit=10,query_filter=QdrantFilter(must=[HasIdCondition(has_id=valid_ids)]))
    end = time.perf_counter()
    qdrant_ids = [hit.id for hit in qdrant_hits.points]
    all_recalls_3["qdrant"].append(calculate_recall(qdrant_ids,gt_ids))
    latencies_3["qdrant"].append(end-start)

    start = time.perf_counter()
    weaviate_res = weaviate_collection.query.near_vector(near_vector=query_vec, limit=10, return_properties=["doc_id"],filters=WeaviateFilter.by_property("doc_id").less_than(1000))
    end = time.perf_counter()
    weaviate_ids = [obj.properties["doc_id"] for obj in weaviate_res.objects]
    all_recalls_3["weaviate"].append(calculate_recall(weaviate_ids,gt_ids))
    latencies_3["weaviate"].append(end-start)

    start =time.perf_counter()
    pinecone_res = pinecone_index.query(vector=query_vec, top_k=10, include_metadata=False, include_values=False, filter={"doc_id": {"$lt": 1000}})
    end = time.perf_counter()
    pinecone_ids = [int(match.id) for match in pinecone_res["matches"]]
    all_recalls_3["pinecone"].append(calculate_recall(pinecone_ids,gt_ids))
    latencies_3["pinecone"].append(end-start)

    start = time.perf_counter()
    chroma_res = chroma_collection.query(query_embeddings=[query_vec], n_results=10, include=[],where={"doc_id": {"$lt": 1000}}) # Only get ID
    end = time.perf_counter()
    chroma_ids = [int(i) for i in chroma_res["ids"][0]]
    all_recalls_3["chroma"].append(calculate_recall(chroma_ids,gt_ids))
    latencies_3["chroma"].append(end-start)

In [40]:
for db, recall in all_recalls_3.items():
    filter_recall = [v for v in recall if v is not None]
    print(f"Average recall of {db}: {100*np.mean(filter_recall):.2f}%")

Average recall of qdrant: 100.00%
Average recall of pinecone: 99.09%
Average recall of milvus: 95.68%
Average recall of weaviate: 100.00%
Average recall of chroma: 91.21%


In [41]:
for db, latency in latencies_3.items():
    p50 = np.percentile(latency, 50)
    p95 = np.percentile(latency, 95)
    p99 = np.percentile(latency, 99)
    mean = np.mean(latency)

    print(f"{db} DB:")
    print(f"Baseline Latency (P99): {p99:.2f} ms")
    print(f"Median Latency (P50): {p50:.2f} ms")
    print(f"Mean Latency (Mean): {mean:.2f} ms")

qdrant DB:
Baseline Latency (P99): 0.27 ms
Median Latency (P50): 0.26 ms
Mean Latency (Mean): 0.26 ms
pinecone DB:
Baseline Latency (P99): 0.29 ms
Median Latency (P50): 0.25 ms
Mean Latency (Mean): 0.25 ms
milvus DB:
Baseline Latency (P99): 0.24 ms
Median Latency (P50): 0.22 ms
Mean Latency (Mean): 0.23 ms
weaviate DB:
Baseline Latency (P99): 0.10 ms
Median Latency (P50): 0.09 ms
Mean Latency (Mean): 0.09 ms
chroma DB:
Baseline Latency (P99): 0.29 ms
Median Latency (P50): 0.25 ms
Mean Latency (Mean): 0.25 ms


## Recall@10 and Latency with filter (level 3)

In [53]:
all_recalls_4 = {"qdrant":[], "pinecone":[], "milvus":[], "weaviate":[], "chroma":[]}
latencies_4 = {"qdrant":[], "pinecone":[], "milvus":[], "weaviate":[], "chroma":[]}


#Query with filter id < 1000
for i in range(len(test_queries)):
    query_vec = list(model.embed([test_queries[i]]))[0].tolist()

    gt_ids = get_ground_truth(np.array(query_vec),10,3)
    
    start = time.perf_counter()
    milvus_results = milvus.search(collection_name="rag_test", data = [query_vec], limit=10,output_fields=['id'],filter="category == 'finance' and is_public == true") #3-D
    end = time.perf_counter()
    milvus_ids = [hit["id"] for hit in milvus_results[0]]
    all_recalls_4["milvus"].append(calculate_recall(milvus_ids,gt_ids))
    latencies_4["milvus"].append(end-start)
    
    start = time.perf_counter()
    qdrant_hits = qdrant.query_points(collection_name="rag_test",query=query_vec,limit=10,query_filter=QdrantFilter(must=[FieldCondition(key="category", match=MatchValue(value="finance")), FieldCondition(key="is_public", match=MatchValue(value=True))]))
    end = time.perf_counter()
    qdrant_ids = [hit.id for hit in qdrant_hits.points]
    all_recalls_4["qdrant"].append(calculate_recall(qdrant_ids,gt_ids))
    latencies_4["qdrant"].append(end-start)

    start = time.perf_counter()
    weaviate_res = weaviate_collection.query.near_vector(near_vector=query_vec, limit=10, return_properties=["doc_id"],filters=WeaviateFilter.by_property("category").equal("finance") & WeaviateFilter.by_property("is_public").equal(True))
    end = time.perf_counter()
    weaviate_ids = [obj.properties["doc_id"] for obj in weaviate_res.objects]
    all_recalls_4["weaviate"].append(calculate_recall(weaviate_ids,gt_ids))
    latencies_4["weaviate"].append(end-start)

    start =time.perf_counter()
    pinecone_res = pinecone_index.query(vector=query_vec, top_k=10, include_metadata=False, include_values=False, filter={"$and": [{"category": {"$eq": "finance"}}, {"is_public": {"$eq": True}}]})
    end = time.perf_counter()
    pinecone_ids = [int(match.id) for match in pinecone_res["matches"]]
    all_recalls_4["pinecone"].append(calculate_recall(pinecone_ids,gt_ids))
    latencies_4["pinecone"].append(end-start)

    start = time.perf_counter()
    chroma_res = chroma_collection.query(query_embeddings=[query_vec], n_results=10, include=[],where={"$and": [{"category": {"$eq": "finance"}}, {"is_public": {"$eq": True}}]})# Only get ID
    end = time.perf_counter()
    chroma_ids = [int(i) for i in chroma_res["ids"][0]]
    all_recalls_4["chroma"].append(calculate_recall(chroma_ids,gt_ids))
    latencies_4["chroma"].append(end-start)

In [54]:
for db, recall in all_recalls_4.items():
    filter_recall = [v for v in recall if v is not None]
    print(f"Average recall of {db}: {100*np.mean(filter_recall):.2f}%")

Average recall of qdrant: 100.00%
Average recall of pinecone: 99.16%
Average recall of milvus: 21.25%
Average recall of weaviate: 100.00%
Average recall of chroma: 97.14%


In [ ]:
print(all_recalls_4["milvus"])
print(len(milvus_ids), milvus_ids[:5]) #Milvus has realy low recall when highly selective filter 

[0.4, 0.3, 0.4, 0.2, 0.0, 0.2, 0.1, 0.2, 0.0, 0.1, 0.2, 0.3, 0.1, 0.3, 0.2, 0.3, 0.1, 0.2, 0.1, 0.2, 0.3, 0.1, 0.2, 0.3, 0.1, 0.0, 0.2, 0.1, 0.3, 0.4, 0.1, 0.1, 0.3, 0.3, 0.3, 0.0, 0.2, 0.3, 0.0, 0.3, 0.0, 0.3, 0.3, 0.2, 0.3, 0.2, 0.3, 0.3, 0.2, 0.2, 0.3, 0.0, 0.2, 0.1, 0.0, 0.2, 0.4, 0.4, 0.4, 0.2, 0.1, 0.2, 0.6, 0.4, 0.3, 0.4, 0.2, 0.3, 0.3, 0.0, 0.2, 0.2, 0.2, 0.0, 0.3, 0.3, 0.2, 0.3, 0.2, 0.1, 0.2, 0.4, 0.1, 0.3, 0.0, 0.2, 0.4, 0.4, 0.2, 0.3, 0.5, 0.2, 0.3, 0.3, 0.1, 0.1, 0.3, 0.0, 0.2, 0.4, 0.3, 0.4, 0.5, 0.3, 0.3, 0.2, 0.2, 0.0, 0.2, 0.2, 0.2, 0.2, 0.0, 0.2, 0.1, 0.3, 0.1, 0.2, 0.3, 0.3, 0.2, 0.0, 0.1, 0.3, 0.3, 0.1, 0.3, 0.1, 0.3, 0.2, 0.2, 0.3, 0.2, 0.3, 0.1, 0.1, 0.2, 0.1, 0.1, 0.0, 0.4, 0.1, 0.1, 0.2, 0.1, 0.0, 0.1, 0.2, 0.2, 0.2, 0.5, 0.4, 0.2, 0.3, 0.3, 0.1, 0.3, 0.1, 0.2, 0.2, 0.3, 0.0, 0.0, 0.1, 0.3, 0.3, 0.1, 0.1, 0.1, 0.3, 0.4, 0.1, 0.1, 0.0, 0.3, 0.2, 0.3, 0.2, 0.4, 0.2, 0.1, 0.2, 0.2, 0.3, 0.4, 0.0, 0.4, 0.1, 0.2, 0.0, 0.1, 0.0, 0.4, 0.2, 0.1, 0.1, 0.2, 0.1, 0.2, 0.2,

In [55]:
for db, latency in latencies_4.items():
    p50 = np.percentile(latency, 50)
    p95 = np.percentile(latency, 95)
    p99 = np.percentile(latency, 99)
    mean = np.mean(latency)

    print(f"{db} DB:")
    print(f"Baseline Latency (P99): {p99:.2f} ms")
    print(f"Median Latency (P50): {p50:.2f} ms")
    print(f"Mean Latency (Mean): {mean:.2f} ms")
    print(f"\n")

qdrant DB:
Baseline Latency (P99): 0.28 ms
Median Latency (P50): 0.27 ms
Mean Latency (Mean): 0.27 ms


pinecone DB:
Baseline Latency (P99): 0.25 ms
Median Latency (P50): 0.24 ms
Mean Latency (Mean): 0.25 ms


milvus DB:
Baseline Latency (P99): 0.26 ms
Median Latency (P50): 0.24 ms
Mean Latency (Mean): 0.25 ms


weaviate DB:
Baseline Latency (P99): 0.10 ms
Median Latency (P50): 0.08 ms
Mean Latency (Mean): 0.08 ms


chroma DB:
Baseline Latency (P99): 0.30 ms
Median Latency (P50): 0.25 ms
Mean Latency (Mean): 0.26 ms




## Ingest Test

In [15]:
import os
import time
import numpy as np
from tqdm import tqdm
from dotenv import load_dotenv

# Load credentials
load_dotenv()

# Configuration
VECTOR_DIM = 768
BATCH_SIZE = 500
TOTAL_VECTORS = 10000  # Start with 10k

# Load base data
try:
    _all_vecs = np.load("msmarco_vectors_768.npy")
    print(f"Loaded base {len(_all_vecs)} vectors.")
except FileNotFoundError:
    print("Error: msmarco_vectors_768.npy not found. Please run the notebook first.")
    exit()

def get_data_batch(count):
    """Generate repeated data to match the target count."""
    indices = np.random.choice(len(_all_vecs), count)
    vectors = _all_vecs[indices].tolist()
    # Dummy metadata
    metadata = [
        {"category": np.random.choice(["finance", "legal", "tech", "medical"]), "is_public": i % 2 == 0}
        for i in range(count)
    ]
    return vectors, metadata

Loaded base 10000 vectors.


In [17]:
# ─────────────────────────────────────────
# DB IGNESTION FUNCTIONS
# ─────────────────────────────────────────

def ingest_qdrant():
    from qdrant_client import QdrantClient
    from qdrant_client.models import Distance, VectorParams, PointStruct
    
    client = QdrantClient(url=os.getenv("qdrant_endpoint"), api_key=os.getenv("qdrant_api_key"), timeout=60)
    if client.collection_exists("ingest_test"):
        client.delete_collection("ingest_test")
    client.create_collection(
        collection_name="ingest_test",
        vectors_config=VectorParams(size=VECTOR_DIM, distance=Distance.COSINE)
    )
    
    start_total = time.perf_counter()
    try:
        for i in tqdm(range(0, TOTAL_VECTORS, BATCH_SIZE), desc="Qdrant Ingest"):
            vectors, metas = get_data_batch(BATCH_SIZE)
            points = [
                PointStruct(id=i + j, vector=v, payload=m)
                for j, (v, m) in enumerate(zip(vectors, metas))
            ]
            client.upload_points(collection_name="ingest_test", points=points, wait=False)
    finally: 
        client.close()
    duration = time.perf_counter() - start_total
    return duration

def ingest_milvus():
    from pymilvus import MilvusClient
    client = MilvusClient(uri=os.getenv("milvus_endpoint"), token=os.getenv("milvus_token"),timeout=60)
    
    if client.has_collection("ingest_test"):
        client.drop_collection("ingest_test")
    client.create_collection(collection_name="ingest_test", dimension=VECTOR_DIM)
    
    start_total = time.perf_counter()
    try:
        for i in tqdm(range(0, TOTAL_VECTORS, BATCH_SIZE), desc="Milvus Ingest"):
            vectors, metas = get_data_batch(BATCH_SIZE)
            data = [
                {"id": i + j, "vector": v, **m}
                for j, (v, m) in enumerate(zip(vectors, metas))
            ]
            client.insert(collection_name="ingest_test", data=data)
    finally:
        client.close()
    duration = time.perf_counter() - start_total
    return duration

def ingest_weaviate():
    import weaviate
    client = weaviate.connect_to_weaviate_cloud(
        cluster_url=os.getenv("weaviate_endpoint"),
        auth_credentials=weaviate.auth.AuthApiKey(os.getenv("weaviate_api_key")),
        additional_config=weaviate.config.AdditionalConfig(timeout=(10, 60))
    )
    if client.collections.exists("IngestTest"): client.collections.delete("IngestTest")
    col = client.collections.create("IngestTest")
    
    start = time.perf_counter()
    try:
        with col.batch.dynamic() as batch:
            for i in tqdm(range(TOTAL_VECTORS), desc="Weaviate"):
                batch.add_object(properties={"c":"t"}, vector=_all_vecs[i%len(_all_vecs)].tolist())
    finally: client.close()
    return time.perf_counter() - start

def ingest_pinecone():
    from pinecone import Pinecone
    pc = Pinecone(api_key=os.getenv("pinecone_api_key"))
    # Note: Assumes index 'test' exists and is empty or we overwrite IDs
    index = pc.Index("test") 
    
    start_total = time.perf_counter()
    for i in tqdm(range(0, TOTAL_VECTORS, BATCH_SIZE), desc="Pinecone Ingest"):
        vectors, metas = get_data_batch(BATCH_SIZE)
        ids = [str(i + j) for j in range(BATCH_SIZE)]
        to_upsert = list(zip(ids, vectors, metas))
        index.upsert(vectors=to_upsert)
    
    duration = time.perf_counter() - start_total
    return duration

def ingest_chroma():
    import chromadb
    client = chromadb.CloudClient(
        api_key=os.getenv("chroma_api_key"),
        tenant=os.getenv("chroma_tenant"),
        database="test"
    )
    if "ingest_test" in [c.name for c in client.list_collections()]:
        client.delete_collection("ingest_test")
    collection = client.create_collection("ingest_test")
    
    start_total = time.perf_counter()
    for i in tqdm(range(0, TOTAL_VECTORS, 300), desc="Chroma Ingest"):
        vectors, metas = get_data_batch(300)
        ids = [str(i + j) for j in range(300)]
        collection.add(ids=ids, embeddings=vectors, metadatas=metas)
    
    duration = time.perf_counter() - start_total
    return duration

In [14]:
# Run tests for 10k vectors
import warnings 
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
results = {}
dbs = {
        "Qdrant": ingest_qdrant,
        "Milvus": ingest_milvus,
        "Weaviate": ingest_weaviate,
        "Pinecone": ingest_pinecone,
        "Chroma": ingest_chroma
    }
    
print(f"\n--- Starting Ingest Benchmark ({TOTAL_VECTORS} vectors) ---")
for name, func in dbs.items():
        try:
            t = func()
            vps = TOTAL_VECTORS / t
            results[name] = {"time": t, "vps": vps}
            print(f"{name}: {t:.2f}s ({vps:.2f} vectors/sec)")
        except Exception as e:
            print(f"{name} failed: {e}")

print("\n--- FINAL RESULTS SUMMARY ---")
print(f"{'DB Name':<12} | {'Time (s)':<10} | {'Speed (vec/s)':<15}")
print("-" * 45)
for name, data in results.items():
        print(f"{name:<12} | {data['time']:<10.2f} | {data['vps']:<15.2f}")


--- Starting Ingest Benchmark (10000 vectors) ---


Qdrant Ingest:  35%|███▌      | 7/20 [01:43<03:25, 15.78s/it]C:\Users\Thanh Nam\AppData\Local\Temp\ipykernel_7868\3756088614.py:25: UserWarning: Batch upload failed 1 times. Retrying...
  client.upload_points(collection_name="ingest_test", points=points, wait=True)
Qdrant Ingest: 100%|██████████| 20/20 [05:06<00:00, 15.31s/it]


Qdrant: 306.29s (32.65 vectors/sec)


Milvus Ingest: 100%|██████████| 20/20 [00:42<00:00,  2.12s/it]


Milvus: 42.45s (235.55 vectors/sec)


...

Weaviate: 9.72s (1028.89 vectors/sec)


Pinecone Ingest: 100%|██████████| 20/20 [03:25<00:00, 10.28s/it]


Pinecone: 205.62s (48.63 vectors/sec)


Chroma Ingest: 100%|██████████| 34/34 [01:04<00:00,  1.89s/it]

Chroma: 64.18s (155.80 vectors/sec)

--- FINAL RESULTS SUMMARY ---
DB Name      | Time (s)   | Speed (vec/s)  
---------------------------------------------
Qdrant       | 306.29     | 32.65          
Milvus       | 42.45      | 235.55         
Weaviate     | 9.72       | 1028.89        
Pinecone     | 205.62     | 48.63          
Chroma       | 64.18      | 155.80         


In [18]:
# Run tests for 100k vectors
TOTAL_VECTORS=100000
import warnings 
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
results = {}
dbs = {
        "Qdrant": ingest_qdrant,
        "Milvus": ingest_milvus,
        "Weaviate": ingest_weaviate,
        "Pinecone": ingest_pinecone,
        "Chroma": ingest_chroma
    }
    
print(f"\n--- Starting Ingest Benchmark ({TOTAL_VECTORS} vectors) ---")
for name, func in dbs.items():
        try:
            t = func()
            vps = TOTAL_VECTORS / t
            results[name] = {"time": t, "vps": vps}
            print(f"{name}: {t:.2f}s ({vps:.2f} vectors/sec)")
        except Exception as e:
            print(f"{name} failed: {e}")

print("\n--- FINAL RESULTS SUMMARY ---")
print(f"{'DB Name':<12} | {'Time (s)':<10} | {'Speed (vec/s)':<15}")
print("-" * 45)
for name, data in results.items():
        print(f"{name:<12} | {data['time']:<10.2f} | {data['vps']:<15.2f}")


--- Starting Ingest Benchmark (100000 vectors) ---


Qdrant Ingest: 100%|██████████| 200/200 [49:25<00:00, 14.83s/it]


Qdrant: 2965.27s (33.72 vectors/sec)


Milvus Ingest: 100%|██████████| 200/200 [08:17<00:00,  2.49s/it]


Milvus: 497.88s (200.85 vectors/sec)


...

Weaviate: 33.28s (3004.73 vectors/sec)


Pinecone Ingest: 100%|██████████| 200/200 [35:19<00:00, 10.60s/it]


Pinecone: 2119.48s (47.18 vectors/sec)


Chroma Ingest: 100%|██████████| 334/334 [10:32<00:00,  1.89s/it]

Chroma: 631.13s (158.45 vectors/sec)

--- FINAL RESULTS SUMMARY ---
DB Name      | Time (s)   | Speed (vec/s)  
---------------------------------------------
Qdrant       | 2965.27    | 33.72          
Milvus       | 497.88     | 200.85         
Weaviate     | 33.28      | 3004.73        
Pinecone     | 2119.48    | 47.18          
Chroma       | 631.13     | 158.45         


## Data visibility latency

In [19]:
#Check the latency when inserting a new vector, how long vectorDB takes to find that vector
query_vec = list(model.embed([test_query["text"]]))[0].tolist()

# --- 1. Qdrant ---
def insert_qdrant(query_vec, test_query):
    from qdrant_client.models import PointStruct
    qdrant.upsert(
        collection_name="rag_test",
        points=[PointStruct(
            id=test_query["id"], 
            vector=query_vec, 
            payload={
                "text": test_query["text"], 
                "category": test_query["category"], 
                "is_public": test_query["is_public"]
            }
        )]
    )

# --- 2. Milvus ---
def insert_milvus(query_vec, test_query):
    milvus.insert(
        collection_name="rag_test",
        data=[{
            "id": test_query["id"], 
            "vector": query_vec, 
            "text": test_query["text"], 
            "category": test_query["category"], 
            "is_public": test_query["is_public"]
        }]
    )

# --- 3. Weaviate (v4) ---
def insert_weaviate(query_vec, test_query):
    weaviate_collection.data.insert(
        properties={
            "doc_id": test_query["id"],
            "text": test_query["text"],
            "category": test_query["category"],
            "is_public": test_query["is_public"]
        },
        vector=query_vec
    )

# --- 4. Pinecone ---
def insert_pinecone(query_vec, test_query):
    pinecone_index.upsert(
        vectors=[(
            str(test_query["id"]), 
            query_vec, 
            {
                "text": test_query["text"], 
                "category": test_query["category"], 
                "is_public": test_query["is_public"], 
                "doc_id": test_query["id"]
            }
        )]
    )

# --- 5. Chroma ---
def insert_chroma(query_vec, test_query):
    chroma_collection.add(
        ids=[str(test_query["id"])],
        embeddings=[query_vec],
        metadatas=[{
            "category": test_query["category"], 
            "is_public": test_query["is_public"], 
            "doc_id": test_query["id"]
        }],
        documents=[test_query["text"]]
    )

# --- 1. Qdrant ---
def search_qdrant(query_vec):
    res = qdrant.query_points(collection_name="rag_test", query=query_vec, limit=1)
    return res.points[0].id if res.points else None
# --- 2. Milvus ---
def search_milvus(query_vec):
    res = milvus.search(collection_name="rag_test", data=[query_vec], limit=1, output_fields=['id'])
    return res[0][0]['id'] if res and res[0] else None
# --- 3. Weaviate (v4) ---
def search_weaviate(query_vec):
    res = weaviate_collection.query.near_vector(near_vector=query_vec, limit=1, return_properties=["doc_id"])
    return res.objects[0].properties["doc_id"] if res.objects else None
# --- 4. Pinecone ---
def search_pinecone(query_vec):
    res = pinecone_index.query(vector=query_vec, top_k=1, include_metadata=False, include_values=False)
    return int(res["matches"][0].id) if res["matches"] else None
# --- 5. Chroma ---
def search_chroma(query_vec):
    res = chroma_collection.query(query_embeddings=[query_vec], n_results=1, include=[])
    return int(res["ids"][0][0]) if res["ids"] and res["ids"][0] else None


In [18]:
import time
results = {}
dbs = {
        "Qdrant": (insert_qdrant,search_qdrant),
        "Milvus": (insert_milvus,search_milvus),
        "Weaviate": (insert_weaviate,search_weaviate),
        "Pinecone": (insert_pinecone,search_pinecone),
        "Chroma": (insert_chroma,search_chroma)
    }
MAX_WAIT_TIME = 60 
for name, func in dbs.items():
        try:
            start = time.perf_counter()
            t = func[0](query_vec,test_query)
            while True:
                result = func[1](query_vec)
                if time.perf_counter() - start >= MAX_WAIT_TIME:
                    print(f"{name} takes too much time")
                if test_query["id"] == result:
                    end = time.perf_counter()
                    break
            results[name] = {"time": end - start}
        except Exception as e:
            print(f"{name} failed: {e}")

print("\n--- FINAL RESULTS SUMMARY ---")
print(f"{'DB Name':<12} | {'Time (s)':<10}")
print("-" * 45)
for name, data in results.items():
        print(f"{name:<12} | {data['time']:<10.2f}")

✅ Qdrant: Inserted doc 10001

--- FINAL RESULTS SUMMARY ---
DB Name      | Time (s)  
---------------------------------------------
Qdrant       | 1.28      
Milvus       | 0.82      
Weaviate     | 0.75      
Pinecone     | 0.83      
Chroma       | 1.09      


## Index Construction Overhead

In [27]:
# Simulate 100 users send requests when inserting more 10,000 vectors into database
import numpy as np 
documents = [item["text"] for item in update_corpus]

embeddings_generator = model.embed(documents,batch_size=128)
all_update_embeddings = np.array(list(embeddings_generator))

np.save("msmarco_vectors_768(2).npy", all_update_embeddings)

In [10]:
#Run this for next time
import numpy as np 
all_update_embeddings = np.load("msmarco_vectors_768(2).npy")

In [16]:
bulk_insert_qdrant(update_corpus,all_update_embeddings)

Qdrant done


In [17]:
bulk_insert_pinecone(update_corpus,all_update_embeddings)

Pinecone done


In [18]:
bulk_insert_chroma(update_corpus,all_update_embeddings)

Chroma done


In [11]:
bulk_insert_weaviate(update_corpus,all_update_embeddings)

Weaviate done


In [20]:
bulk_insert_milvus(update_corpus,all_update_embeddings)

Milvus done
